# Optimizers in TensorFlow

This notebook maps PyTorch optimizers to TensorFlow/Keras equivalents, builds a custom RAdam optimizer, explores LR scheduling approaches, and validates numerical parity with PyTorch.

**Sections:**
1. Setup & TF/PyTorch optimizer mapping
2. TF/Keras optimizer tour
3. Custom RAdam optimizer subclass
4. LR scheduling in TensorFlow (schedule objects, callbacks, manual)
5. Warmup + cosine decay without a built-in combined scheduler
6. Parity check vs. PyTorch Adam
7. Fashion-MNIST optimizer comparison

In [ ]:
# Section 1 — Setup
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

tf.random.set_seed(42)
np.random.seed(42)

## Section 2 — PyTorch → TensorFlow Optimizer Mapping

| PyTorch | TensorFlow/Keras | Key difference |
|---|---|---|
| `torch.optim.SGD` | `tf.keras.optimizers.SGD` | TF `momentum` is raw; PyTorch normalizes by (1−β) |
| `torch.optim.Adam` | `tf.keras.optimizers.Adam` | Same algorithm; different `epsilon` default (1e-7 in TF) |
| `torch.optim.AdamW` | `tf.keras.optimizers.AdamW` | Available in TF 2.12+; same decoupled decay semantics |
| `torch.optim.RMSprop` | `tf.keras.optimizers.RMSprop` | TF `rho` = PyTorch `alpha`; centered option same |
| `torch.optim.Adagrad` | `tf.keras.optimizers.Adagrad` | API nearly identical |
| `torch.optim.Adadelta` | `tf.keras.optimizers.Adadelta` | `rho` same semantics |
| `torch.optim.Adamax` | `tf.keras.optimizers.Adamax` | Same update rule |
| `torch.optim.NAdam` | `tf.keras.optimizers.Nadam` | Same Nesterov-Adam |
| `torch.optim.LBFGS` | No built-in equivalent | Use `tfp.optimizer.lbfgs_minimize` (TensorFlow Probability) |
| `torch.optim.Rprop` | No built-in equivalent | Implement as custom optimizer |
| `torch.optim.ASGD` | No built-in equivalent | Implement via custom callback |
| `torch.optim.SparseAdam` | `tf.keras.optimizers.Adam` | TF handles sparse grads natively with `IndexedSlices` |

In [ ]:
# Section 2 — Quick TF optimizer tour
# Build a simple synthetic dataset
N, D = 1000, 20
X_np = np.random.randn(N, D).astype(np.float32)
W_true = np.random.randn(D, 1).astype(np.float32)
y_np = X_np @ W_true + 0.1 * np.random.randn(N, 1).astype(np.float32)

def build_mlp():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(D,)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1)
    ])

tf_optimizers = [
    ('SGD',       tf.keras.optimizers.SGD(learning_rate=0.01)),
    ('SGD+mom',   tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)),
    ('Adagrad',   tf.keras.optimizers.Adagrad(learning_rate=0.01)),
    ('RMSprop',   tf.keras.optimizers.RMSprop(learning_rate=0.001)),
    ('Adadelta',  tf.keras.optimizers.Adadelta(learning_rate=1.0)),
    ('Adam',      tf.keras.optimizers.Adam(learning_rate=0.001)),
    ('AdamW',     tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.01)),
    ('Adamax',    tf.keras.optimizers.Adamax(learning_rate=0.002)),
    ('Nadam',     tf.keras.optimizers.Nadam(learning_rate=0.002)),
]

tf_results = {}
for name, opt in tf_optimizers:
    model = build_mlp()
    model.compile(optimizer=opt, loss='mse')
    history = model.fit(X_np, y_np, epochs=30, batch_size=64, verbose=0)
    tf_results[name] = history.history['loss']
    print(f'{name:15s}  final loss: {tf_results[name][-1]:.6f}')

plt.figure(figsize=(11, 5))
for name, losses in tf_results.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.yscale('log')
plt.title('TF/Keras Optimizer Convergence')
plt.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## Section 3 — Custom RAdam Optimizer

TensorFlow does not include RAdam out of the box. We subclass `tf.keras.optimizers.Optimizer` and implement the variance-rectified update from Liu et al. 2019.

In [ ]:
# Section 3 — Custom RAdam optimizer in TensorFlow
class RAdam(tf.keras.optimizers.Optimizer):
    """
    Rectified Adam (RAdam) — Liu et al. 2019.
    Falls back to SGD with momentum in early steps when second-moment
    variance is high (rho_t <= 4), then switches to adaptive updates.
    """
    def __init__(self, learning_rate=1e-3, beta_1=0.9, beta_2=0.999,
                 epsilon=1e-8, name='RAdam', **kwargs):
        super().__init__(learning_rate=learning_rate, name=name, **kwargs)
        self.beta_1  = beta_1
        self.beta_2  = beta_2
        self.epsilon = epsilon
        self.rho_inf = 2.0 / (1.0 - beta_2) - 1.0

    def build(self, var_list):
        super().build(var_list)
        self._m  = [self.add_variable_from_reference(v, 'm')  for v in var_list]
        self._v  = [self.add_variable_from_reference(v, 'v')  for v in var_list]

    def update_step(self, gradient, variable, learning_rate):
        lr   = tf.cast(learning_rate, variable.dtype)
        b1   = tf.cast(self.beta_1,   variable.dtype)
        b2   = tf.cast(self.beta_2,   variable.dtype)
        eps  = tf.cast(self.epsilon,  variable.dtype)

        idx  = self._variables.index(variable)
        m, v = self._m[idx], self._v[idx]

        t    = tf.cast(self.iterations + 1, variable.dtype)
        m.assign(b1 * m + (1.0 - b1) * gradient)
        v.assign(b2 * v + (1.0 - b2) * tf.square(gradient))

        m_hat = m / (1.0 - tf.pow(b1, t))
        rho_t = (self.rho_inf
                 - 2.0 * t * tf.pow(b2, t) / (1.0 - tf.pow(b2, t)))

        def adaptive_update():
            rect = tf.sqrt(
                (rho_t - 4) * (rho_t - 2) * self.rho_inf
                / ((self.rho_inf - 4) * (self.rho_inf - 2) * rho_t)
            )
            v_hat = tf.sqrt(v / (1.0 - tf.pow(b2, t))) + eps
            return lr * rect * m_hat / v_hat

        def sgd_update():
            return lr * m_hat

        step = tf.cond(rho_t > 4.0, adaptive_update, sgd_update)
        variable.assign_sub(step)

    def get_config(self):
        config = super().get_config()
        config.update({'beta_1': self.beta_1, 'beta_2': self.beta_2, 'epsilon': self.epsilon})
        return config

# Compare RAdam vs Adam on synthetic data
model_radam = build_mlp()
model_adam2 = build_mlp()
model_radam.compile(optimizer=RAdam(learning_rate=1e-3), loss='mse')
model_adam2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

h_radam = model_radam.fit(X_np, y_np, epochs=50, batch_size=64, verbose=0)
h_adam2 = model_adam2.fit(X_np, y_np, epochs=50, batch_size=64, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(h_radam.history['loss'], label='Custom RAdam')
plt.plot(h_adam2.history['loss'], label='Adam (built-in)')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.yscale('log')
plt.title('Custom RAdam vs Built-in Adam'); plt.legend()
plt.tight_layout(); plt.show()
print('RAdam should match or slightly improve Adam, especially in early epochs.')

## Section 4 — LR Scheduling in TensorFlow

TF has three approaches to LR scheduling:
1. **Schedule objects** — pass a `tf.keras.optimizers.schedules.*` instance as the `learning_rate` arg
2. **ReduceLROnPlateau callback** — metric-driven decay like PyTorch's equivalent
3. **LearningRateScheduler callback** — arbitrary Python function of epoch

In [ ]:
# Section 4 — LR scheduling comparison
EPOCHS = 80

def make_schedule_model(schedule):
    model = build_mlp()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=schedule), loss='mse')
    return model

# 1. CosineDecay schedule object
cosine_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=EPOCHS * (N // 64), alpha=1e-6
)

# 2. ExponentialDecay schedule object
exp_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3, decay_steps=10 * (N // 64), decay_rate=0.9
)

# 3. PolynomialDecay
poly_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=1e-3, decay_steps=EPOCHS * (N // 64), end_learning_rate=1e-7
)

schedules = [
    ('CosineDecay',     cosine_schedule),
    ('ExponentialDecay',exp_schedule),
    ('PolynomialDecay', poly_schedule),
]

# Visualize LR traces
TOTAL_STEPS = EPOCHS * (N // 64)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, sched) in zip(axes, schedules):
    steps = np.arange(TOTAL_STEPS)
    lrs   = [float(sched(s)) for s in steps]
    ax.plot(steps, lrs)
    ax.set_title(name); ax.set_xlabel('Step'); ax.set_ylabel('LR')
plt.suptitle('TF LR Schedule Objects', fontsize=13)
plt.tight_layout(); plt.show()

# Train with each schedule
sched_results = {}
for name, sched in schedules:
    h = make_schedule_model(sched).fit(X_np, y_np, epochs=EPOCHS, batch_size=64, verbose=0)
    sched_results[name] = h.history['loss']

# Callback approach: ReduceLROnPlateau
model_plateau = build_mlp()
model_plateau.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)
h_plateau = model_plateau.fit(X_np, y_np, epochs=EPOCHS, batch_size=64, callbacks=[rlrop], verbose=0)
sched_results['ReduceLROnPlateau'] = h_plateau.history['loss']

plt.figure(figsize=(9, 5))
for name, losses in sched_results.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.yscale('log')
plt.title('TF Scheduler Comparison'); plt.legend(); plt.tight_layout(); plt.show()

## Section 5 — Warmup + Cosine Decay (Custom SequentialLR Equivalent)

In [ ]:
# Section 5 — Manual linear warmup + cosine decay in TF
# TF has no built-in SequentialLR, so we implement it as a custom schedule
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warmup for warmup_steps, then cosine decay to min_lr over remaining steps.
    """
    def __init__(self, max_lr, warmup_steps, total_steps, min_lr=1e-7):
        super().__init__()
        self.max_lr       = max_lr
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr

    def __call__(self, step):
        step  = tf.cast(step, tf.float32)
        ws    = tf.cast(self.warmup_steps, tf.float32)
        ts    = tf.cast(self.total_steps,  tf.float32)
        lr    = tf.cast(self.max_lr,       tf.float32)
        min_lr= tf.cast(self.min_lr,       tf.float32)

        warmup_lr = lr * step / ws
        progress  = (step - ws) / tf.maximum(ts - ws, 1)
        cosine_lr = min_lr + 0.5 * (lr - min_lr) * (1 + tf.cos(np.pi * progress))

        return tf.where(step < ws, warmup_lr, cosine_lr)

    def get_config(self):
        return {'max_lr': self.max_lr, 'warmup_steps': self.warmup_steps,
                'total_steps': self.total_steps, 'min_lr': self.min_lr}

TOTAL_STEPS = 80 * (N // 64)
WARMUP_STEPS = 5 * (N // 64)

wcd = WarmupCosineDecay(max_lr=1e-3, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS)
steps = np.arange(TOTAL_STEPS)
lrs   = [float(wcd(s)) for s in steps]

plt.figure(figsize=(9, 4))
plt.plot(steps, lrs)
plt.axvline(x=WARMUP_STEPS, color='r', linestyle='--', label=f'Warmup ends (step {WARMUP_STEPS})')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.title('Custom WarmupCosineDecay Schedule')
plt.legend(); plt.tight_layout(); plt.show()

# Train with and without warmup
m_wcd  = build_mlp()
m_cos  = build_mlp()
cosine_only = tf.keras.optimizers.schedules.CosineDecay(1e-3, TOTAL_STEPS, alpha=1e-7)
m_wcd.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=wcd),        loss='mse')
m_cos.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_only), loss='mse')

h_wcd = m_wcd.fit(X_np, y_np, epochs=80, batch_size=64, verbose=0)
h_cos = m_cos.fit(X_np, y_np, epochs=80, batch_size=64, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(h_wcd.history['loss'], label='Warmup + Cosine')
plt.plot(h_cos.history['loss'], label='Cosine only (no warmup)')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.yscale('log')
plt.title('Warmup + Cosine vs Cosine Only'); plt.legend()
plt.tight_layout(); plt.show()

## Section 6 — Parity Check: TF Adam vs PyTorch Adam

Both frameworks implement the same Adam update rule but differ in defaults:
- TF `epsilon` default: **1e-7** | PyTorch: **1e-8**
- Match them explicitly for numerical parity.

In [ ]:
# Section 6 — TF vs PyTorch parity (pure numpy implementation for comparison)
# We implement Adam from scratch in numpy and compare against TF Adam

def adam_numpy(X, y, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, epochs=20, batch_size=64):
    """Minimal Adam on single-layer linear regression for verification."""
    np.random.seed(42)
    W = np.zeros((X.shape[1], 1), dtype=np.float64)
    m, v = np.zeros_like(W), np.zeros_like(W)
    losses = []
    t = 0
    for epoch in range(epochs):
        idx = np.random.permutation(len(X))
        ep_loss = 0
        for start in range(0, len(X), batch_size):
            xb = X[idx[start:start+batch_size]]
            yb = y[idx[start:start+batch_size]]
            pred = xb @ W
            loss = np.mean((pred - yb)**2)
            grad = 2 * xb.T @ (pred - yb) / len(xb)
            t += 1
            m = beta1 * m + (1 - beta1) * grad
            v = beta2 * v + (1 - beta2) * grad**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            W -= lr * m_hat / (np.sqrt(v_hat) + eps)
            ep_loss += loss
        losses.append(ep_loss / (len(X) // batch_size))
    return losses

X_lin = np.random.randn(1000, 5).astype(np.float32)
W_lin = np.random.randn(5, 1).astype(np.float32)
y_lin = (X_lin @ W_lin).astype(np.float32)

# TF single-layer model with same settings
model_parity = tf.keras.Sequential([tf.keras.layers.Dense(1, use_bias=False, input_shape=(5,))])
model_parity.layers[0].set_weights([np.zeros((5, 1), dtype=np.float32)])
model_parity.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-8),
    loss='mse'
)
h_parity = model_parity.fit(X_lin, y_lin, epochs=20, batch_size=64, verbose=0, shuffle=True)
tf_losses = h_parity.history['loss']

np_losses = adam_numpy(X_lin.astype(np.float64), y_lin.astype(np.float64))

plt.figure(figsize=(8, 4))
plt.plot(tf_losses, 'b-o', ms=4, label='TF Adam')
plt.plot(np_losses, 'r--s', ms=4, label='NumPy Adam (reference)')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Parity Check: TF Adam vs NumPy Adam (same hyperparams)')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Final TF loss:    {tf_losses[-1]:.8f}')
print(f'Final NumPy loss: {np_losses[-1]:.8f}')
print(f'Difference:       {abs(tf_losses[-1] - np_losses[-1]):.2e}')
print('Note: minor differences from float32 vs float64 and shuffle order are expected.')

## Section 7 — Fashion-MNIST Optimizer Comparison

In [ ]:
# Section 7 — Fashion-MNIST: SGD vs Adam vs AdamW vs RMSprop
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test  = x_test.astype(np.float32)  / 255.0
x_train = x_train[..., np.newaxis]  # add channel dim
x_test  = x_test[..., np.newaxis]

def build_convnet():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])

fmnist_optimizers = [
    ('SGD+momentum',   tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)),
    ('Adam',           tf.keras.optimizers.Adam(learning_rate=1e-3)),
    ('AdamW',          tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)),
    ('RMSprop',        tf.keras.optimizers.RMSprop(learning_rate=1e-3)),
    ('Nadam',          tf.keras.optimizers.Nadam(learning_rate=1e-3)),
]

fmnist_results = {}
FMNIST_EPOCHS = 15
for name, opt in fmnist_optimizers:
    model = build_convnet()
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(
        x_train, y_train,
        epochs=FMNIST_EPOCHS, batch_size=128,
        validation_data=(x_test, y_test),
        verbose=0
    )
    final_val_acc = history.history['val_accuracy'][-1]
    fmnist_results[name] = {
        'train_acc': history.history['accuracy'],
        'val_acc':   history.history['val_accuracy'],
    }
    print(f'{name:20s}  final val accuracy: {final_val_acc:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, r in fmnist_results.items():
    ax1.plot(r['train_acc'], label=name)
    ax2.plot(r['val_acc'],   label=name)
ax1.set_title('Training Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=8)
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(fontsize=8)
plt.suptitle('Fashion-MNIST ConvNet: Optimizer Comparison', fontsize=13)
plt.tight_layout(); plt.show()